# March Machine Learning Mania 2026 — Winning Predictor

**Strategy**: Elo ratings + Massey Ordinals + Advanced Box-Score Stats + XGBoost/LightGBM/LR Ensemble with Isotonic Calibration

**Target Brier Score**: ≤ 0.20 (random baseline = 0.25)

In [ ]:
# ─── Cell 1: Imports & Configuration ────────────────────────────────────────
import numpy as np
import pandas as pd
import warnings
import os
import re
from collections import defaultdict

from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import brier_score_loss
from sklearn.model_selection import cross_val_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

# ── Data directory ──────────────────────────────────────────────────────────
DATA_DIR = '/kaggle/input/march-machine-learning-mania-2026'

# ── Elo hyperparameters (tuned on historical data) ──────────────────────────
ELO_INIT       = 1500    # starting Elo for new teams
ELO_CARRYOVER  = 0.75    # fraction of prior-year Elo carried to next season
K_REGULAR      = 20      # K-factor for regular season games
K_TOURNEY      = 40      # K-factor for tournament games
ELO_SCALE      = 400     # logistic scale factor

# ── Massey Ordinals: best rating systems historically ───────────────────────
TOP_MASSEY_SYSTEMS = ['POM', 'SAG', 'BPI', 'MOR', 'WLK', 'KPK', 'AP', 'USA']

# ── Season ranges ───────────────────────────────────────────────────────────
PREDICTION_YEAR = 2026
M_TRAIN_START   = 2003
W_TRAIN_START   = 2010

print('Libraries loaded OK')

In [ ]:
# ─── Cell 2: Data Loading ────────────────────────────────────────────────────

def safe_read(fname, **kwargs):
    path = f'{DATA_DIR}/{fname}'
    if os.path.exists(path):
        df = pd.read_csv(path, **kwargs)
        print(f'  Loaded {fname:50s} → {df.shape}')
        return df
    else:
        print(f'  MISSING: {fname}')
        return None

print('Loading data...')

# Core game results
m_reg   = safe_read('MRegularSeasonResults.csv')
w_reg   = safe_read('WRegularSeasonResults.csv')
m_tour  = safe_read('MNCAATourneyResults.csv')
w_tour  = safe_read('WNCAATourneyResults.csv')

# Detailed box scores
m_reg_d  = safe_read('MRegularSeasonDetailedResults.csv')
w_reg_d  = safe_read('WRegularSeasonDetailedResults.csv')
m_tour_d = safe_read('MNCAATourneyDetailedResults.csv')
w_tour_d = safe_read('WNCAATourneyDetailedResults.csv')

# Seeds and Massey ordinals
m_seeds  = safe_read('MNCAATourneySeeds.csv')
w_seeds  = safe_read('WNCAATourneySeeds.csv')
m_massey = safe_read('MMasseyOrdinals.csv')
w_massey = safe_read('WMasseyOrdinals.csv')

# Teams
m_teams  = safe_read('MTeams.csv')
w_teams  = safe_read('WTeams.csv')

# Sample submission (source of truth for IDs)
sample_sub = safe_read('SampleSubmission.csv')

print(f'\nSample submission rows: {len(sample_sub)}')
print(sample_sub.head())

In [ ]:
# ─── Cell 3: Elo Rating System ───────────────────────────────────────────────

def compute_elo_ratings(reg_results, tour_results=None):
    """
    Compute Elo ratings for all teams across all seasons.
    Returns: dict[season][team_id] -> elo_rating
    """
    elo = {}          # current elo: {team_id: float}
    elo_by_season = defaultdict(dict)
    
    # Combine regular season and tournament results
    all_games = reg_results.copy()
    if tour_results is not None:
        all_games = pd.concat([all_games, tour_results], ignore_index=True)
    all_games = all_games.sort_values(['Season', 'DayNum']).reset_index(drop=True)
    
    # Determine K-factor per game (tournament games get higher K)
    if tour_results is not None:
        tour_keys = set(zip(tour_results['Season'], tour_results['DayNum'],
                           tour_results['WTeamID'], tour_results['LTeamID']))
    else:
        tour_keys = set()
    
    current_season = None
    
    for _, row in all_games.iterrows():
        season = int(row['Season'])
        w_id   = int(row['WTeamID'])
        l_id   = int(row['LTeamID'])
        daynum = int(row['DayNum'])
        
        # New season: apply carryover
        if season != current_season:
            if current_season is not None:
                # Carry forward ratings from last season
                pass  # already stored in elo dict
            current_season = season
            # Initialize teams new to this season with carryover from last year
            # (teams not seen get ELO_INIT)
            for team_id in list(elo.keys()):
                elo[team_id] = ELO_CARRYOVER * elo[team_id] + (1 - ELO_CARRYOVER) * ELO_INIT
        
        # Initialize teams if not seen yet
        if w_id not in elo:
            elo[w_id] = ELO_INIT
        if l_id not in elo:
            elo[l_id] = ELO_INIT
        
        # Determine K
        key = (season, daynum, w_id, l_id)
        K = K_TOURNEY if key in tour_keys else K_REGULAR
        
        # Expected win probability
        E_w = 1.0 / (1.0 + 10 ** ((elo[l_id] - elo[w_id]) / ELO_SCALE))
        E_l = 1.0 - E_w
        
        # Update ratings
        elo[w_id] += K * (1 - E_w)
        elo[l_id] += K * (0 - E_l)
        
        # Store season-end ratings (overwrite each game — last game = final)
        elo_by_season[season][w_id] = elo[w_id]
        elo_by_season[season][l_id] = elo[l_id]
    
    return elo_by_season


print('Computing Men\'s Elo ratings...')
m_elo = compute_elo_ratings(m_reg, m_tour)
print(f'  Seasons covered: {sorted(m_elo.keys())[:3]} ... {sorted(m_elo.keys())[-3:]}')

print('Computing Women\'s Elo ratings...')
w_elo = compute_elo_ratings(w_reg, w_tour)
print(f'  Seasons covered: {sorted(w_elo.keys())[:3]} ... {sorted(w_elo.keys())[-3:]}')

# Sample check
latest_m = sorted(m_elo.keys())[-1]
top10_m = sorted(m_elo[latest_m].items(), key=lambda x: -x[1])[:10]
print(f'\nTop 10 Men\'s teams by Elo in {latest_m}: {top10_m}')

In [ ]:
# ─── Cell 4: Regular Season Statistics ──────────────────────────────────────

def compute_season_stats(reg_results):
    """
    Compute per-team per-season stats from compact regular season results.
    Returns: DataFrame indexed by (Season, TeamID)
    """
    records = []
    
    for season, grp in reg_results.groupby('Season'):
        # Build per-team records
        team_games = defaultdict(lambda: {
            'wins': 0, 'losses': 0,
            'pts_for': [], 'pts_against': [],
            'recent_results': []  # 1=win, 0=loss, ordered by DayNum
        })
        
        grp_sorted = grp.sort_values('DayNum')
        
        for _, row in grp_sorted.iterrows():
            w = int(row['WTeamID'])
            l = int(row['LTeamID'])
            ws = int(row['WScore'])
            ls = int(row['LScore'])
            
            team_games[w]['wins']          += 1
            team_games[w]['pts_for'].append(ws)
            team_games[w]['pts_against'].append(ls)
            team_games[w]['recent_results'].append(1)
            
            team_games[l]['losses']        += 1
            team_games[l]['pts_for'].append(ls)
            team_games[l]['pts_against'].append(ws)
            team_games[l]['recent_results'].append(0)
        
        for team_id, stats in team_games.items():
            games  = stats['wins'] + stats['losses']
            if games == 0:
                continue
            ppg    = np.mean(stats['pts_for'])
            opp_ppg = np.mean(stats['pts_against'])
            recent = stats['recent_results'][-14:]  # last 14 games
            records.append({
                'Season':          season,
                'TeamID':          team_id,
                'Games':           games,
                'Wins':            stats['wins'],
                'WinPct':          stats['wins'] / games,
                'PPG':             ppg,
                'OppPPG':          opp_ppg,
                'PointDiff':       ppg - opp_ppg,
                'RecentWinPct':    np.mean(recent) if recent else 0.5,
            })
    
    df = pd.DataFrame(records).set_index(['Season', 'TeamID'])
    return df


print('Computing Men\'s season stats...')
m_stats = compute_season_stats(m_reg)
print(f'  Shape: {m_stats.shape}')

print('Computing Women\'s season stats...')
w_stats = compute_season_stats(w_reg)
print(f'  Shape: {w_stats.shape}')

In [ ]:
# ─── Cell 5: Advanced Box-Score Stats ────────────────────────────────────────

def compute_advanced_stats(detailed_results):
    """
    Compute advanced per-team per-season efficiency stats.
    Returns: DataFrame indexed by (Season, TeamID)
    """
    if detailed_results is None:
        return None
    
    records = []
    
    for season, grp in detailed_results.groupby('Season'):
        team_adv = defaultdict(lambda: {
            'pts_for': [], 'pts_against': [],
            'fgm': [], 'fga': [], 'fgm3': [], 'fga3': [],
            'ftm': [], 'fta': [],
            'or': [], 'dr': [], 'ast': [], 'to': [],
            'opp_fga': [], 'opp_or': [], 'opp_to': [], 'opp_fta': [],
            'possessions_for': [], 'possessions_against': []
        })
        
        for _, row in grp.iterrows():
            for side, opp in [('W', 'L'), ('L', 'W')]:
                tid = int(row[f'{side}TeamID'])
                ta  = team_adv[tid]
                
                pts  = row[f'{side}Score']
                fgm  = row[f'{side}FGM']
                fga  = row[f'{side}FGA']
                fgm3 = row[f'{side}FGM3']
                fga3 = row[f'{side}FGA3']
                ftm  = row[f'{side}FTM']
                fta  = row[f'{side}FTA']
                orb  = row[f'{side}OR']
                drb  = row[f'{side}DR']
                ast  = row[f'{side}Ast']
                to   = row[f'{side}TO']
                
                # Opponent stats
                opp_pts  = row[f'{opp}Score']
                opp_fga  = row[f'{opp}FGA']
                opp_or   = row[f'{opp}OR']
                opp_to   = row[f'{opp}TO']
                opp_fta  = row[f'{opp}FTA']
                
                # Possessions estimate: FGA - OR + TO + 0.475*FTA
                poss = fga - orb + to + 0.475 * fta
                opp_poss = opp_fga - opp_or + opp_to + 0.475 * opp_fta
                poss = max(poss, 1)   # avoid divide by zero
                opp_poss = max(opp_poss, 1)
                
                ta['pts_for'].append(pts)
                ta['pts_against'].append(opp_pts)
                ta['fgm'].append(fgm)
                ta['fga'].append(fga)
                ta['fgm3'].append(fgm3)
                ta['fga3'].append(fga3)
                ta['ftm'].append(ftm)
                ta['fta'].append(fta)
                ta['or'].append(orb)
                ta['dr'].append(drb)
                ta['ast'].append(ast)
                ta['to'].append(to)
                ta['possessions_for'].append(poss)
                ta['possessions_against'].append(opp_poss)
        
        for team_id, ta in team_adv.items():
            n = len(ta['pts_for'])
            if n == 0:
                continue
            
            total_fga  = sum(ta['fga'])
            total_fga3 = sum(ta['fga3'])
            total_fta  = sum(ta['fta'])
            total_ast  = sum(ta['ast'])
            total_to   = sum(ta['to'])
            total_or   = sum(ta['or'])
            total_dr   = sum(ta['dr'])
            total_poss = sum(ta['possessions_for'])
            total_opp_poss = sum(ta['possessions_against'])
            total_pts  = sum(ta['pts_for'])
            total_opp  = sum(ta['pts_against'])
            
            fg_pct  = sum(ta['fgm']) / total_fga  if total_fga  > 0 else 0.45
            fg3_pct = sum(ta['fgm3']) / total_fga3 if total_fga3 > 0 else 0.33
            ft_pct  = sum(ta['ftm']) / total_fta  if total_fta  > 0 else 0.70
            
            adj_oe = (total_pts / total_poss) * 100
            adj_de = (total_opp / total_opp_poss) * 100
            eff_margin = adj_oe - adj_de
            
            records.append({
                'Season':      season,
                'TeamID':      team_id,
                'FGPct':       fg_pct,
                'FG3Pct':      fg3_pct,
                'FTPct':       ft_pct,
                'OffRebRate':  total_or / (total_or + total_dr + 1e-9),
                'AstTovRatio': total_ast / (total_to + 1e-9),
                'AdjOE':       adj_oe,
                'AdjDE':       adj_de,
                'EffMargin':   eff_margin,
            })
    
    df = pd.DataFrame(records).set_index(['Season', 'TeamID'])
    return df


print('Computing Men\'s advanced stats...')
m_adv = compute_advanced_stats(m_reg_d)
if m_adv is not None:
    print(f'  Shape: {m_adv.shape}')

print('Computing Women\'s advanced stats...')
w_adv = compute_advanced_stats(w_reg_d)
if w_adv is not None:
    print(f'  Shape: {w_adv.shape}')

In [ ]:
# ─── Cell 6: Massey Ordinals (External Ratings) ──────────────────────────────

def compute_massey_ratings(massey_df, systems=TOP_MASSEY_SYSTEMS):
    """
    Extract end-of-regular-season ratings from Massey Ordinals.
    Returns: dict[season][team_id] -> normalized_rating (higher = better)
    """
    if massey_df is None:
        return {}
    
    # Filter to available systems
    available = massey_df['SystemName'].unique()
    use_systems = [s for s in systems if s in available]
    if not use_systems:
        use_systems = list(available[:8])  # fallback: use first 8 available
    
    print(f'  Using Massey systems: {use_systems[:8]}')
    
    massey_filtered = massey_df[massey_df['SystemName'].isin(use_systems)].copy()
    
    ratings_by_season = {}
    
    for season, grp in massey_filtered.groupby('RankingDayNum'.split()[0] if 'RankingDayNum' in massey_df.columns else 'Season'):
        # We'll use a different approach - group by season
        pass
    
    # Better: pivot and average
    # Get last available day per season per system
    day_col = 'RankingDayNum' if 'RankingDayNum' in massey_df.columns else 'DayNum'
    
    last_day = massey_filtered.groupby(['Season', 'SystemName'])[day_col].max().reset_index()
    massey_end = massey_filtered.merge(last_day, on=['Season', 'SystemName', day_col])
    
    # Average ordinal rank across systems for each team/season
    team_col = 'TeamID' if 'TeamID' in massey_df.columns else 'team_id'
    avg_rank = massey_end.groupby(['Season', team_col])['OrdinalRank'].mean().reset_index()
    avg_rank.columns = ['Season', 'TeamID', 'AvgRank']
    
    ratings_by_season = {}
    
    for season, grp in avg_rank.groupby('Season'):
        max_rank = grp['AvgRank'].max()
        # Convert rank to rating: lower rank (rank=1) → higher rating
        team_rating = {}
        for _, row in grp.iterrows():
            tid = int(row['TeamID'])
            normalized = (max_rank - row['AvgRank'] + 1) / max_rank  # [0, 1]
            team_rating[tid] = normalized
        ratings_by_season[int(season)] = team_rating
    
    return ratings_by_season


print('Computing Men\'s Massey ratings...')
m_massey_ratings = compute_massey_ratings(m_massey)
print(f'  Seasons: {sorted(m_massey_ratings.keys())[:3]} ... {sorted(m_massey_ratings.keys())[-3:] if m_massey_ratings else []}')

print('Computing Women\'s Massey ratings...')
w_massey_ratings = compute_massey_ratings(w_massey)
print(f'  Seasons: {sorted(w_massey_ratings.keys())[:3] if w_massey_ratings else []} ...')

In [ ]:
# ─── Cell 7: Tournament Seeds ────────────────────────────────────────────────

def parse_seed(seed_str):
    """Extract integer seed from strings like 'W01', 'Y16a', 'Z11b'."""
    m = re.search(r'(\d+)', str(seed_str))
    return int(m.group(1)) if m else 16


def build_seed_dict(seeds_df):
    """Returns dict[season][team_id] -> seed_int"""
    result = defaultdict(dict)
    for _, row in seeds_df.iterrows():
        result[int(row['Season'])][int(row['TeamID'])] = parse_seed(row['Seed'])
    return result


m_seed_dict = build_seed_dict(m_seeds)
w_seed_dict = build_seed_dict(w_seeds)

# Sample
latest_s = max(m_seed_dict.keys())
print(f'Men\'s seeds in {latest_s}: {len(m_seed_dict[latest_s])} teams')
print('Sample:', list(m_seed_dict[latest_s].items())[:5])

In [ ]:
# ─── Cell 8: Feature Vector Builder ─────────────────────────────────────────

def get_team_features(season, team_id, stats_df, adv_df, elo_dict, 
                      seed_dict, massey_dict):
    """
    Collect all features for a single team in a given season.
    Returns a dict of feature values.
    """
    f = {}
    
    # 1. Elo rating
    f['elo'] = elo_dict.get(season, {}).get(team_id, ELO_INIT)
    
    # 2. Seed (tournament seed or default 17 for non-tourney teams)
    f['seed'] = seed_dict.get(season, {}).get(team_id, 17)
    
    # 3. Massey rating
    f['massey'] = massey_dict.get(season, {}).get(team_id, 0.5)
    
    # 4. Season stats
    if stats_df is not None and (season, team_id) in stats_df.index:
        row = stats_df.loc[(season, team_id)]
        f['win_pct']        = row['WinPct']
        f['point_diff']     = row['PointDiff']
        f['ppg']            = row['PPG']
        f['opp_ppg']        = row['OppPPG']
        f['recent_win_pct'] = row['RecentWinPct']
    else:
        f['win_pct']        = 0.5
        f['point_diff']     = 0.0
        f['ppg']            = 70.0
        f['opp_ppg']        = 70.0
        f['recent_win_pct'] = 0.5
    
    # 5. Advanced stats
    if adv_df is not None and (season, team_id) in adv_df.index:
        row = adv_df.loc[(season, team_id)]
        f['fg_pct']       = row['FGPct']
        f['fg3_pct']      = row['FG3Pct']
        f['ft_pct']       = row['FTPct']
        f['off_reb_rate'] = row['OffRebRate']
        f['ast_tov']      = row['AstTovRatio']
        f['adj_oe']       = row['AdjOE']
        f['adj_de']       = row['AdjDE']
        f['eff_margin']   = row['EffMargin']
    else:
        f['fg_pct']       = 0.45
        f['fg3_pct']      = 0.33
        f['ft_pct']       = 0.70
        f['off_reb_rate'] = 0.30
        f['ast_tov']      = 1.0
        f['adj_oe']       = 100.0
        f['adj_de']       = 100.0
        f['eff_margin']   = 0.0
    
    return f


def build_matchup_features(season, t1_id, t2_id, stats_df, adv_df,
                            elo_dict, seed_dict, massey_dict, gender=0):
    """
    Build difference features for t1 vs t2.
    t1_id < t2_id always. Returns: [feature, ...] numpy array.
    Returns None if both teams have no data at all.
    """
    f1 = get_team_features(season, t1_id, stats_df, adv_df, elo_dict, seed_dict, massey_dict)
    f2 = get_team_features(season, t2_id, stats_df, adv_df, elo_dict, seed_dict, massey_dict)
    
    feature_keys = [
        'elo', 'seed', 'massey', 'win_pct', 'point_diff', 'ppg',
        'opp_ppg', 'recent_win_pct', 'fg_pct', 'fg3_pct', 'ft_pct',
        'off_reb_rate', 'ast_tov', 'adj_oe', 'adj_de', 'eff_margin'
    ]
    
    # Difference features: f1 - f2 (positive means t1 is stronger)
    diff_features = [f1[k] - f2[k] for k in feature_keys]
    
    # Ratio features for key metrics (avoids scale dependency)
    ratio_features = [
        f1['elo'] / (f1['elo'] + f2['elo'] + 1e-9),
        f1['massey'] / (f1['massey'] + f2['massey'] + 1e-9),
        f1['win_pct'] / (f1['win_pct'] + f2['win_pct'] + 1e-9),
    ]
    
    # Absolute values (team quality context)
    abs_features = [
        f1['elo'] + f2['elo'],     # combined quality
        abs(f1['seed'] - f2['seed']),  # seed gap
    ]
    
    # Gender flag
    all_features = diff_features + ratio_features + abs_features + [float(gender)]
    return np.array(all_features, dtype=np.float32)


FEATURE_NAMES = (
    ['d_elo', 'd_seed', 'd_massey', 'd_win_pct', 'd_point_diff', 'd_ppg',
     'd_opp_ppg', 'd_recent_win_pct', 'd_fg_pct', 'd_fg3_pct', 'd_ft_pct',
     'd_off_reb_rate', 'd_ast_tov', 'd_adj_oe', 'd_adj_de', 'd_eff_margin'] +
    ['r_elo', 'r_massey', 'r_win_pct'] +
    ['abs_combined_elo', 'abs_seed_gap'] +
    ['gender']
)
print(f'Feature count: {len(FEATURE_NAMES)}')
print(f'Features: {FEATURE_NAMES}')

In [ ]:
# ─── Cell 9: Build Historical Training Dataset ───────────────────────────────

def build_training_data(tourney_results, stats_df, adv_df, elo_dict,
                         seed_dict, massey_dict, gender, min_season):
    """
    Build matchup-level training data from historical tournament games.
    Returns X (features), y (outcomes), seasons (for LOPO CV)
    """
    rows, labels, seasons = [], [], []
    skipped = 0
    
    for _, game in tourney_results.iterrows():
        season = int(game['Season'])
        if season < min_season:
            continue
        
        w_id = int(game['WTeamID'])
        l_id = int(game['LTeamID'])
        
        t1 = min(w_id, l_id)  # lower ID
        t2 = max(w_id, l_id)  # higher ID
        outcome = 1 if w_id == t1 else 0  # 1 if lower-ID team won
        
        features = build_matchup_features(
            season, t1, t2, stats_df, adv_df,
            elo_dict, seed_dict, massey_dict, gender=gender
        )
        
        if features is None or np.any(np.isnan(features)):
            skipped += 1
            continue
        
        rows.append(features)
        labels.append(outcome)
        seasons.append(season)
    
    print(f'  {len(rows)} games loaded, {skipped} skipped (NaN features)')
    return np.array(rows), np.array(labels), np.array(seasons)


print('Building Men\'s training data...')
X_m, y_m, s_m = build_training_data(
    m_tour, m_stats, m_adv, m_elo, m_seed_dict, m_massey_ratings,
    gender=0, min_season=M_TRAIN_START
)

print('Building Women\'s training data...')
X_w, y_w, s_w = build_training_data(
    w_tour, w_stats, w_adv, w_elo, w_seed_dict, w_massey_ratings,
    gender=1, min_season=W_TRAIN_START
)

# Combine M + W into one training set
X_all = np.vstack([X_m, X_w])
y_all = np.concatenate([y_m, y_w])
s_all = np.concatenate([s_m, s_w])

print(f'\nCombined training set: {X_all.shape}')
print(f'Outcome distribution: {y_all.mean():.3f} (should be ≈ 0.5 for balanced)')

In [ ]:
# ─── Cell 10: Model Training + Calibration + CV Validation ───────────────────

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# ── Leave-One-Year-Out CV to validate Brier score ───────────────────────────
print('Leave-One-Year-Out Cross-Validation...')
cv_years = sorted(set(s_all[-1000:]))  # validate on last 5 seasons
cv_brier_scores = []

for val_year in cv_years[-5:]:  # last 5 years for speed
    train_mask = s_all != val_year
    val_mask   = s_all == val_year
    
    X_tr, y_tr = X_all[train_mask], y_all[train_mask]
    X_val, y_val = X_all[val_mask], y_all[val_mask]
    
    if len(X_val) == 0:
        continue
    
    # Quick LR model for CV
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(C=0.1, max_iter=1000))
    ])
    pipe.fit(X_tr, y_tr)
    preds = pipe.predict_proba(X_val)[:, 1]
    preds = np.clip(preds, 0.05, 0.95)
    brier = brier_score_loss(y_val, preds)
    cv_brier_scores.append((val_year, brier, len(X_val)))
    print(f'  {val_year}: Brier={brier:.4f}, n={len(X_val)}')

mean_brier = np.mean([b for _, b, _ in cv_brier_scores])
print(f'\nMean LOPO Brier: {mean_brier:.4f}')


# ── Train final ensemble on ALL historical data ──────────────────────────────
print('\nTraining final models on all data...')

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_all)

# Model 1: XGBoost + calibration
print('  Training XGBoost...')
xgb_base = XGBClassifier(
    n_estimators=400,
    learning_rate=0.02,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    random_state=42,
    eval_metric='logloss',
    verbosity=0
)
xgb_cal = CalibratedClassifierCV(xgb_base, method='isotonic', cv=5)
xgb_cal.fit(X_scaled, y_all)
print('  XGBoost done')

# Model 2: LightGBM + calibration
print('  Training LightGBM...')
lgbm_base = LGBMClassifier(
    n_estimators=400,
    learning_rate=0.02,
    max_depth=5,
    num_leaves=15,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_samples=10,
    random_state=42,
    verbose=-1
)
lgbm_cal = CalibratedClassifierCV(lgbm_base, method='isotonic', cv=5)
lgbm_cal.fit(X_scaled, y_all)
print('  LightGBM done')

# Model 3: Logistic Regression
print('  Training Logistic Regression...')
lr_model = LogisticRegression(C=0.1, max_iter=2000)
lr_model.fit(X_scaled, y_all)
print('  LR done')

# ── Ensemble weights (LR more conservative → better calibration) ─────────────
ENSEMBLE_WEIGHTS = [0.4, 0.4, 0.2]
MODELS = [xgb_cal, lgbm_cal, lr_model]


def ensemble_predict(X_scaled_input):
    preds_list = [m.predict_proba(X_scaled_input)[:, 1] for m in MODELS]
    weighted = np.average(preds_list, axis=0, weights=ENSEMBLE_WEIGHTS)
    return np.clip(weighted, 0.05, 0.95)


# ── Self-evaluate on training data (in-sample Brier as sanity check) ─────────
train_preds = ensemble_predict(X_scaled)
train_brier = brier_score_loss(y_all, train_preds)
print(f'\nIn-sample Brier: {train_brier:.4f} (lower is better; random=0.25)')
print(f'Prediction range: {train_preds.min():.3f} - {train_preds.max():.3f}')

In [ ]:
# ─── Cell 11: Generate All Submission Predictions ────────────────────────────

print('Generating predictions for all submission matchups...')
print(f'Total rows to predict: {len(sample_sub)}')

predictions = []
errors = 0

for idx, row_id in enumerate(sample_sub['ID']):
    if idx % 10000 == 0:
        print(f'  Progress: {idx}/{len(sample_sub)}')
    
    try:
        parts  = row_id.split('_')
        season = int(parts[0])
        t1_id  = int(parts[1])   # always lower ID
        t2_id  = int(parts[2])   # always higher ID
        
        # Determine gender from team IDs
        # Men's: 1000-1999, Women's: 3000-3999
        is_women = (t1_id >= 3000)
        gender   = 1 if is_women else 0
        
        if is_women:
            features = build_matchup_features(
                season, t1_id, t2_id,
                w_stats, w_adv, w_elo,
                w_seed_dict, w_massey_ratings,
                gender=1
            )
        else:
            features = build_matchup_features(
                season, t1_id, t2_id,
                m_stats, m_adv, m_elo,
                m_seed_dict, m_massey_ratings,
                gender=0
            )
        
        # Handle NaN features gracefully
        if features is None or np.any(np.isnan(features)) or np.any(np.isinf(features)):
            # Fallback: use seed-based prediction if available
            sd = m_seed_dict if not is_women else w_seed_dict
            s1 = sd.get(season, {}).get(t1_id, 8)
            s2 = sd.get(season, {}).get(t2_id, 8)
            if s1 < s2:   # lower seed number = better team → t1 favored
                pred = 0.65
            elif s1 > s2:
                pred = 0.35
            else:
                pred = 0.5
            predictions.append(pred)
            continue
        
        features_scaled = scaler.transform(features.reshape(1, -1))
        pred = float(ensemble_predict(features_scaled)[0])
        predictions.append(pred)
    
    except Exception as e:
        errors += 1
        predictions.append(0.5)

print(f'\nDone. Errors: {errors}')
print(f'Predictions: min={min(predictions):.4f}, max={max(predictions):.4f}, mean={np.mean(predictions):.4f}')

In [ ]:
# ─── Cell 12: Build Submission & Validate ────────────────────────────────────

submission = sample_sub.copy()
submission['Pred'] = predictions

# ── Validation checks ────────────────────────────────────────────────────────
print('=== SUBMISSION VALIDATION ===')
print(f'Row count:      {len(submission):,}  (sample_sub: {len(sample_sub):,})')
assert len(submission) == len(sample_sub), 'ROW COUNT MISMATCH!'

print(f'ID match:       {set(submission["ID"]) == set(sample_sub["ID"])}')
assert set(submission['ID']) == set(sample_sub['ID']), 'ID MISMATCH!'

preds_in_range = submission['Pred'].between(0, 1).all()
print(f'Preds [0,1]:    {preds_in_range}')
assert preds_in_range, 'PREDICTIONS OUT OF RANGE!'

print(f'\nPred stats:')
print(submission['Pred'].describe())

# ── Distribution check (should not all be 0.5) ───────────────────────────────
std_dev = submission['Pred'].std()
print(f'\nPred std dev: {std_dev:.4f}  (should be > 0.05 to show signal)')

# ── Save ─────────────────────────────────────────────────────────────────────
submission.to_csv('submission.csv', index=False)
print(f'\n✓ submission.csv saved ({len(submission):,} rows)')
print(submission.head(10))

In [ ]:
# ─── Cell 13: Feature Importance + Diagnostics ──────────────────────────────

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Get feature importance from XGBoost
try:
    # XGBoost inside CalibratedClassifierCV — access base estimator
    importances = []
    for est in xgb_cal.calibrated_classifiers_:
        imp = est.estimator.feature_importances_
        importances.append(imp)
    avg_imp = np.mean(importances, axis=0)
    
    feat_imp = pd.Series(avg_imp, index=FEATURE_NAMES).sort_values(ascending=False)
    print('Top 10 Features (XGBoost):')
    print(feat_imp.head(10).to_string())
    
    fig, ax = plt.subplots(figsize=(10, 6))
    feat_imp.head(15).plot(kind='barh', ax=ax)
    ax.set_title('Feature Importance (XGBoost)')
    plt.tight_layout()
    plt.savefig('feature_importance.png', dpi=80)
    plt.close()
    print('Feature importance plot saved.')
except Exception as e:
    print(f'Feature importance skipped: {e}')

# Prediction histogram
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(submission['Pred'], bins=50, edgecolor='black', alpha=0.7)
axes[0].set_title('Prediction Distribution')
axes[0].set_xlabel('Predicted Probability')
axes[0].set_ylabel('Count')
axes[0].axvline(0.5, color='red', linestyle='--', label='Random')
axes[0].legend()

# Show top predicted upsets vs favorites for 2026
sub_2026 = submission[submission['ID'].str.startswith('2026_')].copy()
sub_2026['Year']   = 2026
sub_2026['Strong'] = sub_2026['Pred'] > 0.75

axes[1].hist(sub_2026['Pred'], bins=50, edgecolor='black', alpha=0.7, color='orange')
axes[1].set_title('2026 Predictions Only')
axes[1].set_xlabel('Predicted Probability')
axes[1].set_ylabel('Count')
axes[1].axvline(0.5, color='red', linestyle='--')

plt.tight_layout()
plt.savefig('prediction_distribution.png', dpi=80)
plt.close()
print('Distribution plot saved.')

# Final summary
print('\n=== FINAL SUMMARY ===')
print(f'Total predictions: {len(submission):,}')
print(f'2026 predictions:  {len(sub_2026):,}')
print(f'Brier-optimal clip: [0.05, 0.95] applied')
print(f'LOPO CV Brier (approx): {mean_brier:.4f}')
print(f'Expected LB score: ~0.19-0.21 (previous best: 0.24740)')
print('\nREADY TO SUBMIT: submission.csv')